# 短期记忆 (Short-Term Memory) 完整教程

## 概述

短期记忆是 AI Agent 记忆系统的核心组件，负责维护当前对话的上下文。本教程将深入介绍：

1. **为什么需要短期记忆** - LLM 的无状态特性
2. **Message 数据结构** - 消息的原子单位
3. **四种记忆策略** - 各有优劣的实现方案
4. **实际应用场景** - 如何选择合适的策略
5. **性能对比与可视化** - 直观理解各策略特点

---

## 环境准备

In [ ]:
import sys
sys.path.insert(0, '../src')

from short_term_memory import (
    Message, MessageRole,
    ConversationBuffer, SlidingWindowMemory,
    SummaryMemory, TokenBasedMemory,
    SimpleTokenCounter, SimpleSummarizer,
    create_conversation_memory
)
from datetime import datetime
import time

print("模块导入成功!")

---

## 1. 为什么需要短期记忆？

### 1.1 LLM 的无状态特性

大型语言模型 (LLM) 本身是**无状态**的 - 每次 API 调用都是独立的。

In [ ]:
# 模拟无记忆的 LLM 交互
def stateless_llm(message: str) -> str:
    """模拟无状态 LLM - 每次调用独立"""
    if "名字" in message or "name" in message.lower():
        return "抱歉，我不知道你的名字。"
    return f"收到消息: {message}"

# 问题演示
print("用户: 我叫张三")
print(f"AI: {stateless_llm('我叫张三')}")
print()
print("用户: 我的名字是什么？")
print(f"AI: {stateless_llm('我的名字是什么？')}")
print()
print("❌ 问题: AI 无法记住之前的对话!")

### 1.2 短期记忆的解决方案

通过维护对话历史，将上下文传递给 LLM：

In [ ]:
# 使用短期记忆
memory = ConversationBuffer()

def stateful_llm(user_message: str, memory: ConversationBuffer) -> str:
    """带记忆的 LLM 模拟"""
    memory.add_user_message(user_message)
    
    # 检查历史消息
    history = memory.get_messages()
    for msg in history:
        if "我叫" in msg.content:
            name = msg.content.split("我叫")[1].strip()
            if "名字" in user_message:
                response = f"你的名字是{name}。"
                memory.add_assistant_message(response)
                return response
    
    response = f"收到: {user_message}"
    memory.add_assistant_message(response)
    return response

# 演示
print("用户: 我叫张三")
print(f"AI: {stateful_llm('我叫张三', memory)}")
print()
print("用户: 我的名字是什么？")
print(f"AI: {stateful_llm('我的名字是什么？', memory)}")
print()
print("✅ 解决: AI 现在可以记住对话历史!")

---

## 2. Message 数据结构

### 2.1 创建消息

In [ ]:
# 创建不同角色的消息
system_msg = Message(role=MessageRole.SYSTEM, content="你是一个有帮助的助手。")
user_msg = Message(role=MessageRole.USER, content="请介绍一下 Python 编程语言。")
assistant_msg = Message(role=MessageRole.ASSISTANT, content="Python 是一种高级编程语言...")
tool_msg = Message(role=MessageRole.TOOL, content='{"result": "success"}')

print("=== 消息角色 ===")
for msg in [system_msg, user_msg, assistant_msg, tool_msg]:
    print(f"  {msg.role.value:10} | {msg.content[:40]}...")

### 2.2 消息属性详解

In [ ]:
# 创建带完整属性的消息
msg = Message(
    role=MessageRole.USER,
    content="这是一条重要的消息，请务必记住！",
    importance=0.9,
    metadata={"source": "web", "session_id": "abc123"}
)

print("=== 消息属性 ===")
print(f"ID:        {msg.id}")
print(f"角色:      {msg.role.value}")
print(f"内容:      {msg.content}")
print(f"Token数:   {msg.token_count}")
print(f"重要性:    {msg.importance}")
print(f"时间戳:    {msg.timestamp}")
print(f"元数据:    {msg.metadata}")

### 2.3 消息序列化

In [ ]:
# OpenAI API 格式
openai_format = msg.to_dict()
print("OpenAI 格式:")
print(f"  {openai_format}")

# 从字典恢复
restored = Message.from_dict({"role": "user", "content": "恢复的消息"})
print(f"\n恢复的消息: {restored}")

---

## 3. 策略一: ConversationBuffer (完整缓存)

### 3.1 基本用法

In [ ]:
# 创建带系统消息的缓冲区
buffer = ConversationBuffer(system_message="你是一个专业的 Python 编程助手。")

# 模拟对话
conversations = [
    ("user", "什么是列表推导式？"),
    ("assistant", "列表推导式是 Python 中创建列表的简洁方式..."),
    ("user", "能给个例子吗？"),
    ("assistant", "当然！例如: [x**2 for x in range(10)]"),
    ("user", "如何添加条件过滤？"),
    ("assistant", "可以在末尾添加 if 条件: [x for x in range(10) if x % 2 == 0]"),
]

for role, content in conversations:
    if role == "user":
        buffer.add_user_message(content)
    else:
        buffer.add_assistant_message(content)

print(f"消息总数: {len(buffer)}")
print(f"Token 总数: {buffer.token_count}")

In [ ]:
# 查看所有消息
print("=== 完整对话历史 ===")
for i, msg in enumerate(buffer.get_messages()):
    content = msg.content[:50] + "..." if len(msg.content) > 50 else msg.content
    print(f"{i+1}. [{msg.role.value:9}] {content}")

### 3.2 转换为 API 格式

In [ ]:
# 转换为 OpenAI API 格式
api_messages = buffer.to_openai_messages()

print("=== OpenAI API 格式 ===")
for msg in api_messages:
    content = msg['content'][:40] + "..." if len(msg['content']) > 40 else msg['content']
    print(f"  {{'role': '{msg['role']}', 'content': '{content}'}}")

### 3.3 清空与重置

In [ ]:
print(f"清空前消息数: {len(buffer)}")
buffer.clear()
print(f"清空后消息数: {len(buffer)}")
print(f"系统消息保留: {buffer.get_messages()[0].content if buffer.get_messages() else 'None'}")

---

## 4. 策略二: SlidingWindowMemory (滑动窗口)

### 4.1 基本原理

只保留最近的 k 条消息，旧消息自动丢弃。

```
窗口大小 k=3:

时间 →
[M1] → [M1, M2] → [M1, M2, M3] → [M2, M3, M4] → [M3, M4, M5]
                                  ↑ M1 被丢弃    ↑ M2 被丢弃
```

In [ ]:
# 创建滑动窗口记忆
window = SlidingWindowMemory(window_size=4, system_message="系统提示")

print("=== 滑动窗口演示 ===")
for i in range(8):
    window.add_user_message(f"消息 {i}")
    msgs = [m.content for m in window.get_messages() if m.role != MessageRole.SYSTEM]
    print(f"添加消息 {i} 后: {msgs}")

### 4.2 系统消息保护

In [ ]:
# 系统消息不计入窗口大小
window2 = SlidingWindowMemory(window_size=2, system_message="重要系统指令")

for i in range(5):
    window2.add_user_message(f"用户消息 {i}")

print("=== 系统消息保护 ===")
for msg in window2.get_messages():
    print(f"  [{msg.role.value}] {msg.content}")

print(f"\n总消息数: {len(window2.get_messages())} (1 系统 + 2 窗口)")

### 4.3 动态调整窗口大小

In [ ]:
window3 = SlidingWindowMemory(window_size=5)
for i in range(5):
    window3.add_user_message(f"消息 {i}")

print(f"原始窗口大小: {window3.window_size}")
print(f"消息数: {len(window3.get_messages())}")

# 缩小窗口
window3.set_window_size(2)
print(f"\n调整后窗口大小: {window3.window_size}")
print(f"消息数: {len(window3.get_messages())}")
print(f"保留的消息: {[m.content for m in window3.get_messages()]}")

---

## 5. 策略三: SummaryMemory (摘要记忆)

### 5.1 工作原理

当消息数超过阈值时，将旧消息压缩为摘要。

In [ ]:
# 创建摘要记忆
summary_mem = SummaryMemory(
    max_messages=6,      # 最大消息数
    summarize_count=3,   # 每次摘要的消息数
)

print("=== 摘要记忆演示 ===")
for i in range(10):
    summary_mem.add_user_message(f"这是第 {i} 条消息，包含一些重要信息。")
    print(f"添加消息 {i}: 摘要数={summary_mem.summary_count}, 消息数={len(summary_mem.get_messages())}")

In [ ]:
# 查看当前状态
print("\n=== 当前记忆状态 ===")
for msg in summary_mem.get_messages():
    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"  [{msg.role.value}] {content}")

---

## 6. 策略四: TokenBasedMemory (Token 预算)

### 6.1 精确控制 Token 使用

In [ ]:
# 创建 Token 预算记忆
token_mem = TokenBasedMemory(
    max_tokens=100,       # 总预算
    reserve_tokens=20,    # 预留给回复
    prioritize_recent=True  # 优先保留最近消息
)

print("=== Token 预算演示 ===")
print(f"总预算: {token_mem.max_tokens}")
print(f"有效预算: {token_mem.max_tokens - 20} (扣除预留)")
print()

for i in range(10):
    token_mem.add_user_message(f"这是消息 {i}，包含一些内容用于测试 Token 计数。")
    print(f"消息 {i}: Token={token_mem.token_count}, 可用={token_mem.available_tokens}, 消息数={len(token_mem.get_messages())}")

---

## 7. 策略对比与选择

### 7.1 性能对比

In [ ]:
# 对比不同策略
strategies = {
    "Buffer": ConversationBuffer(),
    "Window(5)": SlidingWindowMemory(window_size=5),
    "Summary": SummaryMemory(max_messages=8, summarize_count=4),
    "Token(200)": TokenBasedMemory(max_tokens=200, reserve_tokens=50),
}

# 添加相同的消息
for i in range(15):
    msg = f"这是第 {i} 条测试消息，用于对比不同策略的行为。"
    for mem in strategies.values():
        mem.add_user_message(msg)

print("=== 策略对比 (添加 15 条消息后) ===")
print(f"{'策略':<12} | {'消息数':>6} | {'Token数':>8}")
print("-" * 35)
for name, mem in strategies.items():
    print(f"{name:<12} | {len(mem.get_messages()):>6} | {mem.token_count:>8}")

### 7.2 选择指南

| 场景 | 推荐策略 | 理由 |
|------|----------|------|
| 短对话 (<20轮) | ConversationBuffer | 简单，保留完整历史 |
| 长对话 (20-100轮) | SlidingWindowMemory | 控制上下文长度 |
| 超长会话 (>100轮) | SummaryMemory | 压缩历史，保留要点 |
| 成本敏感应用 | TokenBasedMemory | 精确控制 API 成本 |
| 实时聊天 | SlidingWindowMemory | 快速响应，最近上下文 |

### 7.3 工厂函数使用

In [ ]:
# 使用工厂函数创建记忆
configs = [
    ("buffer", {}),
    ("sliding_window", {"window_size": 10}),
    ("summary", {"max_messages": 20, "summarize_count": 10}),
    ("token_based", {"max_tokens": 4000, "reserve_tokens": 500}),
]

print("=== 工厂函数创建 ===")
for strategy, kwargs in configs:
    mem = create_conversation_memory(strategy, **kwargs)
    print(f"  {strategy}: {type(mem).__name__}")

---

## 8. 实际应用示例

### 8.1 构建简单聊天机器人

In [ ]:
class SimpleChatBot:
    """简单聊天机器人示例"""
    
    def __init__(self, strategy: str = "sliding_window", **kwargs):
        self.memory = create_conversation_memory(
            strategy,
            system_message="你是一个友好的助手。",
            **kwargs
        )
    
    def chat(self, user_input: str) -> str:
        self.memory.add_user_message(user_input)
        
        # 模拟 LLM 响应 (实际应用中调用 API)
        response = f"收到你的消息: '{user_input[:20]}...'" if len(user_input) > 20 else f"收到: '{user_input}'"
        
        self.memory.add_assistant_message(response)
        return response
    
    def get_context(self):
        return self.memory.to_openai_messages()

# 使用示例
bot = SimpleChatBot("sliding_window", window_size=5)

messages = ["你好！", "今天天气怎么样？", "推荐一本书", "谢谢！"]
for msg in messages:
    response = bot.chat(msg)
    print(f"用户: {msg}")
    print(f"Bot: {response}\n")

---

## 总结

本教程介绍了 AI Agent 短期记忆系统的四种策略：

1. **ConversationBuffer**: 完整保存，适合短对话
2. **SlidingWindowMemory**: 滑动窗口，适合长对话
3. **SummaryMemory**: 摘要压缩，适合超长会话
4. **TokenBasedMemory**: Token 预算，适合成本控制

选择合适的策略取决于：
- 对话预期长度
- 成本敏感度
- 上下文完整性需求